In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

caminho_do_arquivo_train = '/content/drive/MyDrive/agnews/train.csv'
caminho_do_arquivo_test= '/content/drive/MyDrive/agnews/test.csv'

# Lê o arquivo CSV
df = pd.read_csv(caminho_do_arquivo_train)
df_test = pd.read_csv(caminho_do_arquivo_test)

# Exibe as primeiras 5 linhas do arquivo
df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [5]:
df.rename(columns= {"Class Index" : "class", "Title":"title", "Description": "description"}, inplace = True)
df_test.rename(columns= {"Class Index" : "class", "Title":"title", "Description": "description"}, inplace = True)

In [6]:
df['text']  = df['title']+ df['description']
df = df.drop(columns=['title'])
df = df.drop(columns=['description'])


df_test['text']  = df_test['title']+ df_test['description']
df_test = df_test.drop(columns=['title'])
df_test = df_test.drop(columns=['description'])

In [7]:
df['class'] = df['class'].apply(lambda x : x-1)
df_test['class'] = df_test['class'].apply(lambda x : x-1)

#as classes vão do 0 ao 3 agora

# Nlp


In [8]:
# Removendo stopwords do texto


In [9]:
import re
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

In [11]:
def preprocess_text(text):
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # Remove mentions and hashtags
    text = re.sub(r"\@\w+|\#", "", text)
    # Remove punctuation and numbers
    text = re.sub(r"[^A-Za-zÀ-ÿ\s]", "", text)
    # Convert to lowercase
    text = text.lower()
    
    palavras = text.split()
    palavras = [palavra for palavra in palavras if palavra not in stop_words]
    palavras = " ".join(palavras)

    return palavras


In [ ]:
X_train = df['text']
y_train = df['class']

X_test = df_test['text']
y_test = df_test['class']



# Aplicando o vetorizador
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
vetorizador = TfidfVectorizer(max_features=5000)

In [14]:
X_train_tfid= vetorizador.fit_transform(X_train)
X_test_tfid= vetorizador.transform(X_test)

In [15]:
X_train_tfid

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3145399 stored elements and shape (120000, 5000)>

In [16]:
#agora, ja que vou usar uma mlp, preciso converter a matriz esparsa em um array numpy,e dps converter para um tensor pytorch
X_train_dense = X_train_tfid.toarray()
X_test_dense = X_test_tfid.toarray()

In [17]:
import torch
torch.__version__

'2.11.0+cpu'

In [18]:
print("oi")

oi


In [19]:
X_train_tensor = torch.from_numpy(X_train_dense).float()

y_train_tensor = torch.from_numpy(y_train.values).long()


X_test_tensor = torch.from_numpy(X_train_dense).float()

y_test_tensor = torch.from_numpy(y_train.values).long()

In [20]:

from torch.utils.data import DataLoader, TensorDataset

meu_dataset_train = TensorDataset(X_train_tensor, y_train_tensor)
meu_dataset_test = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(meu_dataset_train, batch_size=32, shuffle=True)

test_loader = DataLoader(meu_dataset_train, batch_size=32, shuffle=False)

In [21]:
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"
device
class Neural(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.layer_1 = nn.Linear(in_features=5000, out_features=128)
        self.relu = nn.ReLU() 
        self.layer_2 = nn.Linear(in_features=128, out_features=4)     
    
    def forward(self, x):        
        # Passa pela camada 1, aplica a não linearidade, e depois vai para a saída
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x


model_0 = Neural().to(device)
model_0



Neural(
  (layer_1): Linear(in_features=5000, out_features=128, bias=True)
  (relu): ReLU()
  (layer_2): Linear(in_features=128, out_features=4, bias=True)
)

In [22]:
# Create loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_0.parameters(), 
                            lr=0.1) 

In [23]:
epochs = 5 

for epoch in range(epochs):    
    train_loss = 0.0
    train_acc = 0.0    
    
    model_0.train() 

    for batch, (X, y) in enumerate(train_loader):
        X, y = X.to(device), y.to(device)
        # 1. Forward pass
        y_pred = model_0(X)
        # 2. Calcula a perda
        loss = loss_fn(y_pred, y)   
        train_loss += loss.item() 
        # 3. Zera os gradientes
        optimizer.zero_grad()
        # 4. Backward pass
        loss.backward()
        # 5. Atualiza os pesos
        optimizer.step()
        # AJUSTE 2 e 3: Pegamos o maior valor direto no y_pred especificando a dim=1
        y_pred_class = torch.argmax(y_pred, dim=1)        
        # Calcula quantos acertos o lote atual teve e acumula
        acertos_do_batch = (y_pred_class == y).sum().item()
        train_acc += acertos_do_batch / len(y)
    # Ao final de cada época, tiramos a média dividindo pelo total de lotes (batches)
    total_batches = len(train_loader)
    loss_medio = train_loss / total_batches
    acc_media = train_acc / total_batches
    print(f"Época {epoch+1}/{epochs} | Perda Média: {loss_medio:.4f} | Acurácia Média: {acc_media:.4f}")


Época 1/5 | Perda Média: 0.7367 | Acurácia Média: 0.7726
Época 2/5 | Perda Média: 0.3105 | Acurácia Média: 0.8997
Época 3/5 | Perda Média: 0.2714 | Acurácia Média: 0.9100
Época 4/5 | Perda Média: 0.2519 | Acurácia Média: 0.9147
Época 5/5 | Perda Média: 0.2392 | Acurácia Média: 0.9179


In [24]:

model_0.eval()
for epoch in range(epochs):
    test_loss = 0.0
    test_acc = 0.0        
    with torch.inference_mode():
        for batch, (X, y) in enumerate(test_loader): 
            X, y = X.to(device), y.to(device)          
            
            test_logits = model_0(X)            
            # Calcula a perda (use sempre os logits aqui)
            loss = loss_fn(test_logits, y)
            test_loss += loss.item() # .item() para não estourar a memória
            
            # Previsão das classes (0, 1, 2 ou 3)
            test_pred = torch.softmax(test_logits, dim=1).argmax(dim=1)
            
            # 3. COMPLETOU AQUI: Calcula os acertos do lote atual
            acertos_do_batch = (test_pred == y).sum().item()
            test_acc += acertos_do_batch / len(y)
            
    # Calcula as médias finais da época de teste
    total_test_batches = len(test_loader)
    loss_teste_medio = test_loss / total_test_batches
    acc_teste_media = test_acc / total_test_batches
    
    print(f"[TESTE] Época {epoch+1}/{epochs} | Perda: {loss_teste_medio:.4f} | Acurácia: {acc_teste_media:.4f}")


[TESTE] Época 1/5 | Perda: 0.2168 | Acurácia: 0.9263
[TESTE] Época 2/5 | Perda: 0.2168 | Acurácia: 0.9263
[TESTE] Época 3/5 | Perda: 0.2168 | Acurácia: 0.9263
[TESTE] Época 4/5 | Perda: 0.2168 | Acurácia: 0.9263
[TESTE] Época 5/5 | Perda: 0.2168 | Acurácia: 0.9263
